# Soup QA — Stage 4: два реальных запуска на T4

**Статус этого notebook: PREPARED. Ни одна ячейка обучения здесь ещё не выполнена.**

Откройте `Среда выполнения → Сменить среду выполнения → T4 GPU`. Бесплатный T4 может быть недоступен; платные ресурсы не нужны. Выполняйте ячейки по одной, сверху вниз. Не используйте «Выполнить всё» до проверки настроек.

При любой ошибке остановите последующие этапы и выполните последнюю ячейку архива. Не удаляйте исходные логи и не перезапускайте неудачное обучение поверх них. Для нового эксперимента сначала скачайте архив, затем начните новую чистую сессию.

Подготовлены 40 синтетических обучающих входов и 8 отдельных контрольных входов. Это тест воспроизводимости, не проверка полезности модели.

## 1. Загрузите `stage4-inputs.zip`
Notebook открыт отдельно. Здесь загрузите только подготовленный архив файлов этапа 4; `.venv`, старые логи и веса загружать не нужно.

In [ ]:
from google.colab import files
from pathlib import Path
import os, sys, json, zipfile, hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Выберите один stage4-inputs.zip")
archive_name = next(iter(uploaded))
PROJECT = Path.cwd() / 'soup-stage4'
if PROJECT.exists():
    raise RuntimeError('Каталог уже существует. Сначала сохраните прежний эксперимент.')
PROJECT.mkdir()
with zipfile.ZipFile(archive_name) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Небезопасный путь в архиве')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
manifest = json.loads((PROJECT/'bundle-manifest.json').read_text())
for name, expected in manifest['sha256'].items():
    if hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Хеш не совпал: ' + name)
from scripts.repro_common import COLAB_LOGS, unique_dir, run_logged
PY = str(PROJECT/'.repro-venv/bin/python')
def step(label, module, arguments=(), interpreter=None, allowed=(0,)):
    attempt = unique_dir(COLAB_LOGS, 'notebook-' + label)
    result = run_logged([interpreter or PY, '-m', module, *arguments], attempt/'command', timeout=2400)
    for stream in ('stdout.txt', 'stderr.txt'):
        text = (attempt/'command'/stream).read_text(errors='replace')
        print(stream + ' (последние 6000 символов; оригинал полностью сохранён):')
        print(text[-6000:])
    if result['exit_code'] not in allowed or result['termination']:
        raise RuntimeError('Шаг завершился ошибкой. Остановитесь и выполните ячейку архива.')
    return result
print('Входные файлы проверены. Обучение ещё не запускалось.')


## 2. Проверьте план и допуски ДО обучения
Оба YAML должны различаться только `output`. Прочитайте заранее установленную политику: она не гарантирует битовое совпадение и не означает SHIP. Если вы не согласны с допусками, остановитесь и верните вопрос на review до запусков.

In [ ]:
print((PROJECT/'configs/repro-a.yaml').read_text())
print((PROJECT/'configs/repro-b.yaml').read_text())
print((PROJECT/'configs/repro-experiment.json').read_text())
print((PROJECT/'configs/repro-policy.json').read_text())


## 3. Проверка T4 и установка фиксированного окружения
Создаётся отдельная Python 3.11.16 среда. Скачиваются пакеты CUDA/PyTorch; это может занять несколько минут и несколько ГБ. Эта ячейка не обучает модель. При ошибке установки не меняйте версии наугад — скачайте архив логов.

In [ ]:
step('setup', 'scripts.repro_colab_setup', interpreter=sys.executable)


## 4. Загрузка только закреплённой ревизии маленькой модели
Загрузятся SmolLM2-135M и tokenizer (ограничение 400 MiB). Проверяются GPU, версии, архитектура и хеши всех файлов. Большая TinyLlama не используется.

In [ ]:
step('download', 'scripts.repro_workflow', ['download'])


## 5. Предварительная проверка в настоящем Colab окружении
Это dry-run, а не обучение. Ожидаются 32 train samples, 8 val samples и код 0 для обоих YAML.

In [ ]:
step('preflight', 'scripts.repro_workflow', ['preflight'])


## 6. Реальное обучение A
В новом процессе выполняется `soup train --config configs/repro-a.yaml --yes`. Ожидаются 16 шагов, сохранённый LoRA adapter и завершённая запись Soup. Один exit 0 не считается достаточным. После выполнения можно скачать промежуточный архив последней ячейкой, не сбрасывая runtime.

In [ ]:
step('train-a', 'scripts.repro_workflow', ['train', '--arm', 'a'])


## 7. Реальное независимое обучение B
Не перезапускайте runtime между A и B. GPU UUID, версии и параметры должны совпасть. Запускается новый процесс, без resume, с другим output. Если A не завершилось, эта ячейка остановится.

In [ ]:
step('train-b', 'scripts.repro_workflow', ['train', '--arm', 'b'])


## 8. Фиксированная оценка A
Сохранённый adapter заново загружается с той же базовой моделью. Измеряются loss на 8 отдельных примерах и greedy ответы. Оценка проходит на CPU в float32, чтобы уменьшить шум отдельного GPU inference.

In [ ]:
step('eval-a', 'scripts.repro_workflow', ['evaluate', '--arm', 'a'])


## 9. Та же оценка B
Второй отдельный процесс, те же входы, seed, tokenizer и режим оценки.

In [ ]:
step('eval-b', 'scripts.repro_workflow', ['evaluate', '--arm', 'b'])


## 10. Независимое сравнение
Код 0 — только `WITHIN_PREDECLARED_SCREEN`, 2 — расхождение требует review, 3 — `NOT_RUN` из-за отсутствующих доказательств. Ни один статус не является программным SHIP. Все числа остаются в comparison.json; не меняйте допуски после просмотра.

In [ ]:
step('compare', 'scripts.compare_repro', ['--output', 'logs/stage-4/colab/comparison.json'], allowed=(0,2,3))
print((COLAB_LOGS/'comparison.json').read_text()[:12000])


## 11. Сохранить ВСЕ результаты — выполняйте также после ошибки
Архив включает полные stdout/stderr и exit codes, окружение, конфигурации, входные данные, все сохранённые adapters/checkpoints, loss history, SQLite и сравнение, если оно состоялось. Базовая модель и установленная среда исключены. Скачайте также выполненный notebook: `Файл → Скачать → Скачать IPYNB`. Верните архив и notebook для анализа; не публикуйте веса в Git.

In [ ]:
from scripts.repro_workflow import archive
archive()
result_archive = sorted(PROJECT.glob('stage4-results-*.zip'))[-1]
print('Скачивается:', result_archive.name)
files.download(str(result_archive))
